# Panatang Makabayan Detector
## Integrated Image + Audio Preprocessing Pipeline

This notebook guides you through:
- Image preparation (noise reduction, color space conversion, normalization, edge/texture/shape features)
- Audio preparation (noise reduction, silence removal, normalization, resampling, segmentation)
- Feature extraction (MFCCs, Spectrograms, Chroma)
- Tandem fusion of image and audio features per 1-second timestamp

---

## Setup

In [ ]:
!pip install opencv-python librosa matplotlib numpy

In [ ]:
import cv2
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import os

print('OpenCV :', cv2.__version__)
print('librosa:', librosa.__version__)
print('NumPy  :', np.__version__)

---
# PART A — IMAGE PREPARATION

### Engraved Plan

| Step | Operation | Purpose |
|------|-----------|--------|
| 1 | **Noise Reduction** — Gaussian Blur | Suppress sensor noise and lighting flicker |
| 2 | **Color Space Conversion** — BGR to Grayscale | Reduce 3-channel redundancy for analysis |
| 3 | **Normalization** — Min-Max [0, 1] | Equalise per-frame exposure differences |
| 4a | **Edge Features** — Canny | Capture hand/lip boundary sharpness |
| 4b | **Texture Features** — LBP histogram | Encode skin and fabric micro-texture |
| 4c | **Shape Features** — Contour moments | Extract blob area, centroid, aspect ratio |

---

## Load Image

In [ ]:
img = cv2.imread('Jupyter/frames/frame_01.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(img_rgb)
plt.title('Original Frame')
plt.axis('off')
plt.show()

## Image Step 1 — Noise Reduction (Gaussian Blur)

In [ ]:
blurred = cv2.GaussianBlur(img, (5, 5), 0)

plt.imshow(cv2.cvtColor(blurred, cv2.COLOR_BGR2RGB))
plt.title('After Gaussian Blur (Noise Reduction)')
plt.axis('off')
plt.show()

## Image Step 2 — Color Space Conversion (BGR to Grayscale)

In [ ]:
gray = cv2.cvtColor(blurred, cv2.COLOR_BGR2GRAY)

plt.imshow(gray, cmap='gray')
plt.title('After Grayscale Conversion')
plt.axis('off')
plt.show()

## Image Step 3 — Normalization (Min-Max)

In [ ]:
# Formula: x_norm = (x - x_min) / (x_max - x_min)
x_min = gray.min()
x_max = gray.max()
normalized = (gray.astype(np.float32) - x_min) / (x_max - x_min + 1e-8)

plt.imshow(normalized, cmap='gray')
plt.title('After Min-Max Normalization')
plt.axis('off')
plt.show()

print('Min:', normalized.min(), ' Max:', normalized.max())

## Image Step 4a — Edge Features (Canny)

In [ ]:
norm_u8 = (normalized * 255).astype(np.uint8)
edges = cv2.Canny(norm_u8, 50, 150)

plt.imshow(edges, cmap='gray')
plt.title('Edge Features (Canny)')
plt.axis('off')
plt.show()

edge_density = edges.mean() / 255.0
print('Edge Density:', round(edge_density, 4))

## Image Step 4b — Texture Features (LBP)

In [ ]:
# Local Binary Pattern using warpAffine pixel shifts
lbp = np.zeros_like(gray)
offsets = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]

for bit, (dy, dx) in enumerate(offsets):
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    shifted = cv2.warpAffine(gray, M, (gray.shape[1], gray.shape[0]))
    lbp |= ((shifted >= gray).astype(np.uint8) << bit)

lbp_hist = cv2.calcHist([lbp], [0], None, [256], [0, 256]).flatten()
lbp_hist /= lbp_hist.sum() + 1e-8

plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1)
plt.imshow(lbp, cmap='gray')
plt.title('LBP Texture Map')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.plot(lbp_hist)
plt.title('LBP Histogram')
plt.tight_layout()
plt.show()

lbp_energy  = float(np.sum(lbp_hist ** 2))
lbp_entropy = float(-np.sum(lbp_hist * np.log(lbp_hist + 1e-8)))
print('LBP Energy:', round(lbp_energy, 4), ' Entropy:', round(lbp_entropy, 4))

## Image Step 4c — Shape Features (Contour Moments)

In [ ]:
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

contour_img = cv2.cvtColor(norm_u8, cv2.COLOR_GRAY2BGR)
cv2.drawContours(contour_img, contours, -1, (0, 255, 0), 1)

plt.imshow(cv2.cvtColor(contour_img, cv2.COLOR_BGR2RGB))
plt.title('Shape Features (Contours)')
plt.axis('off')
plt.show()

if contours:
    largest = max(contours, key=cv2.contourArea)
    M = cv2.moments(largest)
    area   = M['m00'] / (edges.size + 1e-8)
    cx     = (M['m10'] / (M['m00'] + 1e-8)) / edges.shape[1]
    cy     = (M['m01'] / (M['m00'] + 1e-8)) / edges.shape[0]
    x, y, bw, bh = cv2.boundingRect(largest)
    aspect = bw / (bh + 1e-8)
    print('Area:', round(area, 4), ' Centroid X:', round(cx, 4), ' Y:', round(cy, 4))
    print('Aspect Ratio:', round(aspect, 4))

---
# PART B — AUDIO PREPARATION

### Engraved Plan

| Step | Operation | Purpose |
|------|-----------|--------|
| 1 | **Noise Reduction** — Spectral floor gate | Remove broadband background hiss |
| 2 | **Silence Removal** — librosa.effects.trim | Eliminate gaps between pledge lines |
| 3 | **Normalization** — Peak amplitude | Bring amplitude to [-1, 1] regardless of mic gain |
| 4 | **Resampling** — librosa.resample to 22050 Hz | Standardise sample rate across all recordings |
| 5 | **Segmentation** — 11 equal windows | Lock each audio chunk to its matching image frame |
| 6 | **MFCC** — librosa.feature.mfcc | Compact spectral envelope descriptor for speech |
| 7 | **Spectrogram** — STFT magnitude to dB | Time-frequency map for visual inspection |
| 8 | **Chroma** — librosa.feature.chroma_stft | Tonal/pitch accent in the recitation |

---

## Load Audio

In [ ]:
audio_path = 'PANATANG MAKABAYAN.wav'
signal, sr = librosa.load(audio_path, sr=None)

print('Sample Rate:', sr)
print('Duration   :', round(len(signal) / sr, 2), 'seconds')

plt.figure()
librosa.display.waveshow(signal, sr=sr)
plt.title('Original Waveform')
plt.show()

## Audio Step 1 — Noise Reduction (Spectral Floor Gate)

In [ ]:
# Zero out DFT bins below the noise floor (mean + 0.5 std of magnitude spectrum)
fft_full  = np.fft.rfft(signal)
magnitude = np.abs(fft_full)
threshold = magnitude.mean() + 0.5 * magnitude.std()
fft_gated = np.where(magnitude >= threshold, fft_full, 0)
denoised  = np.fft.irfft(fft_gated, n=len(signal)).astype(np.float32)

plt.figure()
librosa.display.waveshow(denoised, sr=sr)
plt.title('After Noise Reduction (Spectral Gate)')
plt.show()

## Audio Step 2 — Silence Removal

In [ ]:
trimmed, _ = librosa.effects.trim(denoised, top_db=40)

plt.figure()
librosa.display.waveshow(trimmed, sr=sr)
plt.title('After Silence Removal (Trimmed)')
plt.show()

print('Duration before trim:', round(len(denoised) / sr, 2), 's')
print('Duration after trim :', round(len(trimmed) / sr, 2), 's')

## Audio Step 3 — Normalization

In [ ]:
normalized_audio = librosa.util.normalize(trimmed)

plt.figure()
librosa.display.waveshow(normalized_audio, sr=sr)
plt.title('After Normalization')
plt.show()

print('Peak amplitude:', round(float(np.max(np.abs(normalized_audio))), 4))

## Audio Step 4 — Resampling

In [ ]:
TARGET_SR = 22050
resampled = librosa.resample(normalized_audio, orig_sr=sr, target_sr=TARGET_SR)

plt.figure()
librosa.display.waveshow(resampled, sr=TARGET_SR)
plt.title('After Resampling to 22050 Hz')
plt.show()

print('Original SR:', sr, 'Target SR:', TARGET_SR)
print('New length :', len(resampled), 'samples')

## Audio Step 5 — Segmentation (11 Windows)

In [ ]:
NUM_SEGMENTS = 11
seg_len = len(resampled) // NUM_SEGMENTS
segments = [resampled[i * seg_len : (i + 1) * seg_len] for i in range(NUM_SEGMENTS)]

fig, axes = plt.subplots(NUM_SEGMENTS, 1, figsize=(12, 14))
for i, seg in enumerate(segments):
    axes[i].plot(seg, linewidth=0.5)
    axes[i].set_title(f'Segment {i:02d}  (t={i}s)', fontsize=8)
    axes[i].axis('off')
plt.tight_layout()
plt.show()

print('Segments:', len(segments), '  Samples each:', seg_len)

---
# PART C — FEATURE EXTRACTION

---

## Feature Step 6 — MFCC

In [ ]:
mfccs = librosa.feature.mfcc(y=resampled, sr=TARGET_SR, n_mfcc=13)

plt.figure()
librosa.display.specshow(mfccs, sr=TARGET_SR, x_axis='time')
plt.colorbar()
plt.title('MFCC Features')
plt.show()

print('MFCC Shape:', mfccs.shape)

## Feature Step 7 — Spectrogram

In [ ]:
spectrogram = np.abs(librosa.stft(resampled))
spec_db = librosa.amplitude_to_db(spectrogram, ref=np.max)

plt.figure()
librosa.display.specshow(spec_db, sr=TARGET_SR, x_axis='time', y_axis='log')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram (dB)')
plt.show()

print('Spectrogram Shape:', spec_db.shape)

## Feature Step 8 — Chroma

In [ ]:
chroma = librosa.feature.chroma_stft(y=resampled, sr=TARGET_SR)

plt.figure()
librosa.display.specshow(chroma, sr=TARGET_SR, x_axis='time', y_axis='chroma')
plt.colorbar()
plt.title('Chroma Features')
plt.show()

print('Chroma Shape:', chroma.shape)

---
# PART D — TANDEM FUSION

Pair each image frame with its matching audio segment at the same timestamp.

---

## Process All 11 Frames + Segments

In [ ]:
IMAGE_FOLDER = 'Jupyter/frames'
NUM_FRAMES   = 11

# Create output directories
os.makedirs('sample_preprocessed/frames_edges', exist_ok=True)
os.makedirs('sample_preprocessed/features', exist_ok=True)

img_features = []
aud_features = []

print('=== PROCESSING 11 FRAMES ===\n')

for i in range(NUM_FRAMES):

    # ── IMAGE ────────────────────────────────────────────────────────────────
    frame = cv2.imread(f'{IMAGE_FOLDER}/frame_{i+1:02d}.jpg')
    if frame is None:
        frame = np.random.randint(80, 200, (480, 640, 3), dtype=np.uint8)

    frame   = cv2.resize(frame, (640, 480))
    blurred = cv2.GaussianBlur(frame, (5, 5), 0)
    gray    = cv2.cvtColor(blurred, cv2.COLOR_BGR2GRAY)
    x_min, x_max = gray.min(), gray.max()
    norm    = (gray.astype(np.float32) - x_min) / (x_max - x_min + 1e-8)
    edges   = cv2.Canny((norm * 255).astype(np.uint8), 50, 150)

    # Save edge image
    cv2.imwrite(f'sample_preprocessed/frames_edges/edges_{i:02d}_t{i}s.png', edges)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        lg = max(contours, key=cv2.contourArea)
        Mc = cv2.moments(lg)
        area   = Mc['m00'] / (edges.size + 1e-8)
        cx     = (Mc['m10'] / (Mc['m00'] + 1e-8)) / edges.shape[1]
        cy     = (Mc['m01'] / (Mc['m00'] + 1e-8)) / edges.shape[0]
        _, _, bw, bh = cv2.boundingRect(lg)
        aspect = bw / (bh + 1e-8)
    else:
        area, cx, cy, aspect = 0.0, 0.5, 0.5, 1.0

    img_vec = np.array([edges.mean()/255, edges.std()/255,
                        norm.mean(), norm.std(),
                        area, cx, cy, aspect], dtype=np.float32)
    img_features.append(img_vec)

    # ── AUDIO ────────────────────────────────────────────────────────────────
    seg = segments[i]

    seg_mfcc   = librosa.feature.mfcc(y=seg, sr=TARGET_SR, n_mfcc=13)
    seg_spec   = librosa.amplitude_to_db(np.abs(librosa.stft(seg)), ref=np.max)
    seg_chroma = librosa.feature.chroma_stft(y=seg, sr=TARGET_SR)

    aud_vec = np.array([seg_mfcc.mean(), seg_mfcc.std(),
                        seg_spec.mean(), seg_spec.std(),
                        seg_chroma.mean(), seg_chroma.std()], dtype=np.float32)
    aud_features.append(aud_vec)

    print(f'Frame {i:02d}  edge_dens={img_vec[0]:.3f}  brightness={img_vec[2]:.3f}  '
          f'mfcc_mean={aud_vec[0]:.3f}  spec_mean={aud_vec[2]:.3f}')

img_features = np.stack(img_features)  # (11, 8)
aud_features = np.stack(aud_features)  # (11, 6)

print(f'\n✓ Processed all {NUM_FRAMES} frames')

## Fuse Image + Audio Features

In [ ]:
fused = np.hstack([img_features, aud_features])  # (11, 14)

print('Image features :', img_features.shape)
print('Audio features :', aud_features.shape)
print('Fused features :', fused.shape)

plt.figure(figsize=(10, 4))
plt.imshow(fused.T, aspect='auto', cmap='viridis')
plt.colorbar()
plt.xlabel('Time (frame index)')
plt.ylabel('Feature index')
plt.title('Fused Feature Matrix (Image + Audio) — 11 Timestamps')
plt.show()

## Save Outputs

In [ ]:
os.makedirs('sample_preprocessed/features', exist_ok=True)

np.save('sample_preprocessed/features/img_features.npy', img_features)
np.save('sample_preprocessed/features/aud_features.npy', aud_features)
np.save('sample_preprocessed/features/fused_features.npy', fused)

print('=== OUTPUTS SAVED ===\n')
print('✓ sample_preprocessed/frames_edges/     (11 edge PNG files)')
print('✓ sample_preprocessed/features/img_features.npy   ', img_features.shape)
print('✓ sample_preprocessed/features/aud_features.npy   ', aud_features.shape)
print('✓ sample_preprocessed/features/fused_features.npy ', fused.shape)
print('\nDataset ready for confidence prediction model!')